# **FINE-TUNING MODEL CNN-BiGRU-CTC (TAHAP 2)**

Notebook ini melatih ulang model pengenal produk FinSense di atas korpus Tahap 1
— korpus yang dipanen dari rekaman asli lewat forced alignment, lalu diperkaya
augmentasi yang meniru kondisi warung dan rekaman browser.

**Yang diperbaiki di sini:** model v11 dilatih hampir seluruhnya pada kalimat
penuh yang bersih. Akibatnya ia melemah pada dua hal yang justru sering terjadi
di lapangan — ucapan yang berhenti di nama produk saja, dan audio yang kotor
(derau, gema, kompresi Opus dari browser). Korpus Tahap 1 berisi keduanya.

**Pendekatan: fine-tuning, bukan melatih dari nol.** Model v11 sudah mengenali
121 produk dengan baik pada kondisi bersih; yang kurang adalah adaptasi ke
bentuk ucapan dan kondisi akustik baru. Melatih ulang dari nol di atas 14 jam
audio berisiko membuang kemampuan yang sudah ada. Pada uji awal, loss validasi
turun dari **76.4 ke 24.8** hanya dengan 240 sampel — arsitekturnya tidak
bermasalah, distribusi datanyalah yang selama ini tidak cocok.

**Arsitektur sengaja TIDAK diubah.** Dengan menjaga bentuk masukan dan keluaran
tetap sama, model hasil notebook ini bisa langsung menggantikan v11 di
`serve_model.py` tanpa mengubah kode layanan sama sekali — cukup ganti satu
baris path.

## **Prasyarat di Kaggle**

Notebook ini butuh dua dataset ter-attach:

| dataset | isi | dipakai untuk |
|---|---|---|
| korpus Tahap 1 | `manifest*.csv` + folder `wav/` | data latih & validasi |
| model v11 | `finsense_ctc_v11.keras` + `char_vocab.json` | titik awal fine-tuning |

Aktifkan **GPU** di Settings → Accelerator (P100 atau T4). Tanpa GPU satu epoch
bisa berjam-jam.

# **IMPORT LIBRARY**

Tahap ini memuat pustaka yang diperlukan untuk pemrosesan audio, pembentukan pipeline data, serta pelatihan model.

In [ ]:
import os, csv, json, glob, time, random
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import tensorflow as tf
import keras
import librosa
import matplotlib.pyplot as plt

print('TensorFlow :', tf.__version__)
print('Keras      :', keras.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU        :', [g.name for g in gpus] if gpus else 'TIDAK ADA — aktifkan di Settings > Accelerator')

**Insight:**

Pustaka berhasil dimuat. Pemeriksaan GPU dilakukan di awal karena pelatihan
model rekuren seperti BiGRU sangat lambat di CPU — tanpa akselerator, satu epoch
atas 11.250 sampel dapat memakan waktu berjam-jam.

# **KONFIGURASI**

Tahap ini menetapkan lokasi data, lokasi keluaran, serta hyperparameter
pelatihan. Lokasi dataset dideteksi otomatis agar notebook tetap berjalan
meskipun nama dataset di Kaggle berbeda.

In [ ]:
def cari_dataset(*pola):
    """Cari direktori dataset di /kaggle/input berdasarkan berkas penanda."""
    for p in pola:
        hit = glob.glob(f'/kaggle/input/**/{p}', recursive=True)
        if hit:
            return os.path.dirname(hit[0])
    return None

KORPUS = cari_dataset('manifest_train.csv', 'manifest.csv')
MODEL_DIR = cari_dataset('finsense_ctc_v11.keras')
VOCAB_PATH = (glob.glob('/kaggle/input/**/char_vocab.json', recursive=True) or [None])[0]
KELUARAN = '/kaggle/working'

assert KORPUS, 'Dataset korpus tidak ditemukan — attach dataset yang berisi manifest_train.csv'
assert MODEL_DIR and VOCAB_PATH, 'Dataset model v11 tidak ditemukan'

WAV_DIR = os.path.join(KORPUS, 'wav')
print('korpus     :', KORPUS)
print('model awal :', MODEL_DIR)
print('keluaran   :', KELUARAN)

# ── Hyperparameter ────────────────────────────────────────────────
BATCH_SIZE    = 32      # aman untuk P100/T4 pada masukan 800x80
EPOCHS        = 30
LR_AWAL       = 1e-4    # kecil, karena ini fine-tuning bukan latih dari nol
LR_MINIMUM    = 1e-6
PATIENCE_STOP = 6       # epoch tanpa perbaikan sebelum berhenti
PATIENCE_LR   = 3       # epoch tanpa perbaikan sebelum LR diturunkan
SEED          = 42

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

**Insight:**

`LR_AWAL` sengaja diset kecil (`1e-4`). Pada fine-tuning, learning rate sebesar
pelatihan dari nol akan merusak bobot yang sudah baik sebelum model sempat
beradaptasi — gejalanya loss melonjak di epoch pertama lalu tidak pernah pulih.

Perlu dicatat, *mixed precision* sengaja tidak digunakan. Model ini hanya
berukuran ~2 juta parameter sehingga keuntungan kecepatannya tipis, sedangkan
perhitungan CTC pada presisi float16 rentan menghasilkan nilai tak hingga.

# **LOAD DATA**

Tahap ini memuat manifest korpus beserta kosakata karakter. Manifest sudah
dipisah menjadi data latih dan validasi pada tahap pembentukan korpus, dengan
pengelompokan berdasarkan rekaman sumber sehingga tidak ada potongan dari
rekaman yang sama muncul di kedua sisi.

In [ ]:
def baca_manifest(nama):
    with open(os.path.join(KORPUS, nama), encoding='utf-8') as f:
        return list(csv.DictReader(f))

train_rows = baca_manifest('manifest_train.csv')
val_rows   = baca_manifest('manifest_val.csv')

with open(VOCAB_PATH, encoding='utf-8') as f:
    V = json.load(f)
char2idx = V['char2idx']
idx2char = V['idx2char']
BLANK    = V['blank_idx']
CFG = {k: V[k] for k in ('sample_rate','n_mels','n_fft','win_length','hop_length','max_frames')}

print(f'klip latih    : {len(train_rows):,}')
print(f'klip validasi : {len(val_rows):,}')
print(f'kelas karakter: {len(char2idx)} + 1 blank (indeks {BLANK})')
print('konfigurasi audio:', CFG)

**Insight:**

Kosakata karakter dimuat dari berkas milik model v11, bukan dibentuk ulang dari
korpus. Ini penting: indeks karakter harus persis sama dengan yang dipakai saat
model dilatih pertama kali, sebab lapisan keluaran model memetakan indeks
tersebut satu per satu. Membentuk ulang kosakata berisiko menggeser pemetaan dan
membuat bobot lama tidak lagi bermakna.

# **EKSPLORASI KORPUS**

Tahap ini memeriksa komposisi korpus sebelum pelatihan dimulai, meliputi
sebaran varian ucapan, sebaran durasi, serta proporsi klip yang teraugmentasi.
Pemeriksaan ini penting untuk memastikan korpus benar-benar memuat bentuk ucapan
sepotong yang menjadi sasaran perbaikan.

In [ ]:
import collections

komposisi = collections.Counter(r['variant'] for r in train_rows)
augment   = collections.Counter('bersih' if r['augment'] == 'clean' else 'teraugmentasi'
                                for r in train_rows)
durasi    = np.array([float(r['duration']) for r in train_rows])

print('Komposisi varian ucapan (data latih):')
for v, n in komposisi.most_common():
    print(f'  {v:16s} {n:6,d}  ({100*n/len(train_rows):4.1f}%)')

print('\nKondisi akustik:')
for k, n in augment.most_common():
    print(f'  {k:16s} {n:6,d}  ({100*n/len(train_rows):4.1f}%)')

print(f'\nDurasi: min {durasi.min():.2f}s | median {np.median(durasi):.2f}s | '
      f'maks {durasi.max():.2f}s | total {durasi.sum()/3600:.1f} jam')

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(durasi, bins=40, color='#4C78A8')
ax[0].set_title('Sebaran durasi klip'); ax[0].set_xlabel('detik'); ax[0].set_ylabel('jumlah')
v, n = zip(*komposisi.most_common())
ax[1].barh(list(v)[::-1], list(n)[::-1], color='#F58518')
ax[1].set_title('Komposisi varian ucapan'); ax[1].set_xlabel('jumlah klip')
plt.tight_layout(); plt.show()

**Insight:**

Korpus didominasi varian `produk`, yaitu klip yang hanya berisi nama produk
tanpa kalimat pembungkus. Inilah bentuk ucapan yang paling kurang terwakili di
data latih model v11, sehingga porsinya sengaja dibuat paling besar.

Sekitar 79% klip telah melalui augmentasi. Proporsi ini menjaga keseimbangan:
cukup besar agar model belajar tahan terhadap gangguan, namun tetap menyisakan
klip bersih sebagai acuan sinyal tanpa gangguan.

# **PIPELINE DATA**

Tahap ini membangun pipeline yang mengubah berkas audio menjadi pasangan
(*log-mel spectrogram*, label karakter) yang siap dilatih.

Ekstraksi fitur sengaja memakai `librosa` dengan parameter yang identik dengan
jalur inferensi di `ai-service/ctc_lexicon.py`. Perbedaan sekecil apa pun antara
fitur saat pelatihan dan saat pelayanan akan membuat model bekerja baik di
notebook namun buruk di aplikasi — dan selisih semacam itu sangat sulit dilacak
setelah model ter-deploy.

In [ ]:
MAX_LABEL = 120   # label terpanjang di korpus 115 karakter

def ekstrak_logmel(samples):
    """log-mel ternormalisasi, di-pad ke max_frames. Identik dengan jalur inferensi."""
    S = librosa.feature.melspectrogram(
        y=samples, sr=CFG['sample_rate'], n_mels=CFG['n_mels'], n_fft=CFG['n_fft'],
        hop_length=CFG['hop_length'], win_length=CFG['win_length'])
    S_db = librosa.power_to_db(S, ref=np.max).T
    S_db = ((S_db - S_db.mean()) / (S_db.std() + 1e-8)).astype(np.float32)
    T = S_db.shape[0]
    if T < CFG['max_frames']:
        return np.vstack([S_db, np.zeros((CFG['max_frames'] - T, CFG['n_mels']), np.float32)])
    return S_db[:CFG['max_frames']]


def encode_label(teks):
    """Teks -> indeks karakter, di-pad dengan -1 agar panjangnya bisa dihitung ulang."""
    lab = np.full(MAX_LABEL, -1, np.int32)
    ids = [char2idx[c] for c in teks if c in char2idx]
    lab[:len(ids)] = ids
    return lab


def buat_dataset(rows, batch_size, shuffle, cache_name=None):
    files = [r['filename'] for r in rows]
    texts = [r['transcript'] for r in rows]

    def _muat(f, t):
        y, _ = librosa.load(os.path.join(WAV_DIR, f.numpy().decode()), sr=CFG['sample_rate'])
        return ekstrak_logmel(y), encode_label(t.numpy().decode())

    ds = tf.data.Dataset.from_tensor_slices((files, texts))
    ds = ds.map(lambda f, t: tf.py_function(_muat, [f, t], [tf.float32, tf.int32]),
                num_parallel_calls=tf.data.AUTOTUNE)
    # Model dibangun dengan masukan bernama 'log_mel'. Memberi tensor telanjang
    # tetap berjalan, tapi Keras memperingatkannya di setiap epoch dan membuat
    # keluaran notebook sulit dibaca.
    ds = ds.map(lambda a, b: ({'log_mel': tf.ensure_shape(a, (CFG['max_frames'], CFG['n_mels']))},
                              tf.ensure_shape(b, (MAX_LABEL,))))
    # Fitur di-cache setelah epoch pertama. Augmentasi akustik sudah dilakukan
    # saat korpus dibentuk, jadi fitur bersifat tetap dan aman di-cache; variasi
    # antar-epoch tetap ada lewat SpecAugment yang bekerja di dalam model.
    if cache_name:
        ds = ds.cache(os.path.join(KELUARAN, cache_name))
    if shuffle:
        ds = ds.shuffle(min(len(files), 4096), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


train_ds = buat_dataset(train_rows, BATCH_SIZE, shuffle=True,  cache_name='cache_train')
val_ds   = buat_dataset(val_rows,   BATCH_SIZE, shuffle=False, cache_name='cache_val')

contoh_fitur, contoh_label = next(iter(val_ds))
print('bentuk batch fitur :', contoh_fitur['log_mel'].shape)
print('bentuk batch label :', contoh_label.shape)
print('panjang label nyata:', tf.reduce_sum(tf.cast(contoh_label >= 0, tf.int32), axis=1).numpy()[:8])

**Insight:**

Label di-*pad* menggunakan nilai `-1`, bukan `0`. Ini bukan pilihan kosmetik:
indeks `0` pada kosakata ini adalah karakter spasi, sehingga memakainya sebagai
*padding* akan membuat fungsi loss menghitung spasi palsu di ujung setiap label.
Dengan `-1`, panjang label sebenarnya dapat dihitung ulang secara andal di dalam
fungsi loss.

Fitur di-cache ke disk setelah epoch pertama sehingga epoch berikutnya tidak
perlu mengulang ekstraksi log-mel. Ini aman dilakukan karena augmentasi akustik
sudah dibekukan ke dalam berkas audio pada tahap pembentukan korpus; variasi
antar-epoch tetap diperoleh dari SpecAugment yang bekerja di dalam model.

# **ARSITEKTUR MODEL**

Tahap ini memuat model v11 sebagai titik awal fine-tuning. Arsitekturnya tidak
diubah sama sekali: CNN satu dimensi tiga lapis sebagai pengekstrak fitur
lokal, dua lapis BiGRU untuk konteks temporal dua arah, lalu lapisan softmax
per-frame atas 28 kelas (26 huruf, spasi, dan satu *blank* milik CTC).

Satu hal yang perlu perhatian khusus: lapisan **SpecAugment** di dalam model.
Berkas `.keras` menyimpan konfigurasinya, namun definisi kelasnya harus
disediakan ulang saat model dimuat. Pada layanan inferensi, kelas itu sengaja
dibuat sebagai *pass-through*. Untuk pelatihan, kelas tersebut harus benar-benar
melakukan penutupan (*masking*) — bila tidak, regularisasi ini hilang diam-diam
tanpa satu pun pesan galat.

In [ ]:
@keras.saving.register_keras_serializable(package='finsense', name='SpecAugment')
class SpecAugment(keras.layers.Layer):
    """Menutup sebagian pita frekuensi dan rentang waktu secara acak saat pelatihan.

    Model dipaksa mengenali kata tanpa bergantung pada satu pita frekuensi atau
    satu penggal waktu tertentu. Saat inferensi lapisan ini tidak melakukan apa
    pun, sehingga keluaran model tetap deterministik.
    """

    def __init__(self, freq_mask=12, time_mask=30, n_freq=2, n_time=2, **kwargs):
        super().__init__(**kwargs)
        self.freq_mask, self.time_mask = freq_mask, time_mask
        self.n_freq, self.n_time = n_freq, n_time

    def call(self, inputs, training=None):
        if not training:
            return inputs
        x = inputs
        bentuk = tf.shape(x)
        T, F = bentuk[1], bentuk[2]
        for _ in range(self.n_time):
            t  = tf.random.uniform([], 0, self.time_mask + 1, tf.int32)
            t0 = tf.random.uniform([], 0, tf.maximum(T - t, 1), tf.int32)
            x = x * tf.concat([tf.ones([t0]), tf.zeros([t]), tf.ones([T - t0 - t])], 0)[None, :, None]
        for _ in range(self.n_freq):
            f  = tf.random.uniform([], 0, self.freq_mask + 1, tf.int32)
            f0 = tf.random.uniform([], 0, tf.maximum(F - f, 1), tf.int32)
            x = x * tf.concat([tf.ones([f0]), tf.zeros([f]), tf.ones([F - f0 - f])], 0)[None, None, :]
        return x

    def get_config(self):
        cfg = super().get_config()
        cfg.update(freq_mask=self.freq_mask, time_mask=self.time_mask,
                   n_freq=self.n_freq, n_time=self.n_time)
        return cfg


model = keras.models.load_model(
    os.path.join(MODEL_DIR, 'finsense_ctc_v11.keras'), compile=False)
model.summary()

# Verifikasi SpecAugment benar-benar aktif saat training dan mati saat inferensi.
uji = np.ones((2, CFG['max_frames'], CFG['n_mels']), np.float32)
beda_infer = not np.allclose(model({'log_mel': uji}, training=False).numpy(),
                             model({'log_mel': uji}, training=False).numpy())
beda_train = not np.allclose(model({'log_mel': uji}, training=False).numpy(),
                             model({'log_mel': uji}, training=True).numpy())
print(f'\nSpecAugment aktif saat training  : {beda_train}  (harus True)')
print(f'SpecAugment mati saat inferensi  : {not beda_infer}  (harus True)')
assert beda_train and not beda_infer, 'SpecAugment tidak terdaftar dengan benar'

**Insight:**

Model berhasil dimuat dengan sekitar 2 juta parameter, dan pemeriksaan
menunjukkan SpecAugment berperilaku sebagaimana mestinya: mengacak masukan saat
pelatihan, namun tidak mengubah apa pun saat inferensi.

Pemeriksaan ini dijadikan `assert` karena kegagalannya bersifat senyap. Bila
kelas SpecAugment tidak terdaftar dengan benar, Keras akan tetap memuat model
tanpa keluhan, pelatihan tetap berjalan, dan satu-satunya gejala adalah model
yang lebih cepat *overfitting* tanpa alasan yang jelas.

# **FUNGSI LOSS CTC**

Tahap ini mendefinisikan fungsi kerugian *Connectionist Temporal Classification*.
CTC memungkinkan model dilatih tanpa mengetahui posisi setiap karakter di dalam
audio: ia menjumlahkan peluang seluruh penyelarasan yang sah antara urutan
karakter target dan urutan frame keluaran model.

In [ ]:
@keras.saving.register_keras_serializable(package='finsense', name='ctc_loss')
def ctc_loss(y_true, y_pred):
    """y_true: (batch, MAX_LABEL) di-pad -1. y_pred: (batch, T, kelas) softmax.

    Didaftarkan sebagai serializable karena Keras menyimpan konfigurasi compile
    ke dalam berkas .keras. Tanpa pendaftaran ini, setiap checkpoint yang
    dihasilkan ModelCheckpoint hanya bisa dimuat dengan compile=False — dan
    melanjutkan pelatihan dari checkpoint menjadi mustahil.
    """
    # Keras mengecast target menjadi float32; kembalikan ke int32 sebelum dipakai.
    y_true = tf.cast(y_true, tf.int32)

    panjang_label = tf.reduce_sum(tf.cast(y_true >= 0, tf.int32), axis=1)
    label = tf.where(y_true < 0, tf.zeros_like(y_true), y_true)

    batch = tf.shape(y_pred)[0]
    frame = tf.shape(y_pred)[1]

    # Lapisan keluaran model sudah softmax, sedangkan tf.nn.ctc_loss menunggu
    # logits dan menerapkan log_softmax sendiri. Untuk probabilitas yang sudah
    # ternormalisasi, log_softmax(log p) == log p, sehingga log-probabilitas
    # dapat langsung diberikan tanpa mengubah arsitektur model.
    logits = tf.math.log(tf.maximum(y_pred, 1e-9))

    return tf.nn.ctc_loss(
        labels=label, logits=logits,
        label_length=panjang_label, logit_length=tf.fill([batch], frame),
        logits_time_major=False, blank_index=BLANK)


# Uji cepat: loss harus berupa angka berhingga, bukan inf atau nan.
uji_loss = ctc_loss(tf.cast(contoh_label, tf.float32), model(contoh_fitur, training=False))
print('loss per sampel (8 pertama):', np.round(uji_loss.numpy()[:8], 1))
assert np.isfinite(uji_loss.numpy()).all(), 'loss tak hingga — periksa panjang label'

**Insight:**

Fungsi loss didaftarkan sebagai *serializable*. Tanpa itu, checkpoint yang
dihasilkan hanya dapat dimuat dengan `compile=False`, sehingga pelatihan tidak
bisa dilanjutkan dari titik simpan — persoalan yang baru terasa justru saat
sesi Kaggle terputus di tengah jalan.

Nilai loss awal terhitung berhingga untuk seluruh sampel, menandakan setiap
label masih muat di dalam jumlah frame keluaran model. Ini syarat mutlak CTC:
bila sebuah label lebih panjang daripada jumlah frame yang tersedia, tidak ada
satu pun penyelarasan yang sah dan loss menjadi tak hingga.

Besarnya nilai loss awal justru informatif — ia menunjukkan seberapa jauh model
v11 dari korpus baru ini, dan menjadi pembanding untuk menilai keberhasilan
fine-tuning.

# **METRIK EVALUASI**

Tahap ini menyiapkan metrik yang benar-benar mencerminkan keberhasilan sistem.

Loss CTC berguna untuk memandu optimisasi, tetapi ia bukan ukuran yang dipakai
pengguna. Yang menentukan pengalaman kasir adalah **apakah produk yang dikenali
benar** — dan produk ditentukan bukan dari transkrip mentah, melainkan dari
penyekoran seluruh 121 nama produk terhadap keluaran model memakai algoritma
maju CTC, persis seperti yang dilakukan `ai-service/ctc_lexicon.py` di layanan.

Dengan memakai metrik yang sama seperti produksi, angka yang terlihat selama
pelatihan adalah angka yang akan benar-benar diterima pengguna.

In [ ]:
# ── Daftar produk diturunkan dari korpus itu sendiri ──────────────
# Varian 'produk' berisi nama produk polos, sehingga daftar lengkapnya bisa
# dibaca langsung dari manifest tanpa berkas tambahan.
semua_rows = train_rows + val_rows
PRODUK = sorted({r['transcript'] for r in semua_rows if r['variant'] == 'produk'})
print(f'produk dalam korpus: {len(PRODUK)}')

_enc = [np.array([char2idx[c] for c in p if c in char2idx], np.int32) for p in PRODUK]
_panjang = np.array([max(len(e), 1) for e in _enc], np.float32)


def _label_sparse(seqs, baris_dipakai):
    dipakai = [seqs[i] for i in baris_dipakai]
    panjang = np.fromiter((len(s) for s in dipakai), np.int64, len(dipakai))
    nilai   = np.concatenate(dipakai).astype(np.int32)
    baris   = np.repeat(np.arange(len(dipakai), dtype=np.int64), panjang)
    kolom   = np.concatenate([np.arange(n, dtype=np.int64) for n in panjang])
    return tf.SparseTensor(tf.constant(np.stack([baris, kolom], 1)),
                           tf.constant(nilai),
                           tf.constant([len(dipakai), int(panjang.max())], tf.int64))


def _frame_minimum(seq):
    """Frame minimum yang dibutuhkan CTC: satu per karakter, plus blank antar huruf kembar."""
    return int(seq.size + np.count_nonzero(seq[1:] == seq[:-1])) if seq.size else 0


def produk_terpilih(probs):
    """Skor seluruh nama produk terhadap satu posteriorgram, kembalikan yang terbaik.

    Label WAJIB berupa SparseTensor. Dengan label dense, tf.nn.ctc_loss memakai
    implementasi berbasis while_loop yang puluhan kali lebih lambat.
    """
    T = int(probs.shape[0])
    butuh = np.array([_frame_minimum(e) for e in _enc])
    layak = np.flatnonzero(butuh <= T)
    if layak.size == 0:
        return None
    logp = tf.math.log(tf.constant(probs[:, None, :], tf.float32) + 1e-9)
    nll = tf.nn.ctc_loss(
        labels=_label_sparse(_enc, layak),
        logits=tf.tile(logp, [1, int(layak.size), 1]),
        label_length=None, logit_length=tf.fill([int(layak.size)], T),
        logits_time_major=True, blank_index=BLANK).numpy()
    return PRODUK[layak[int(np.argmin(nll))]]


def decode_greedy(probs):
    ids = np.argmax(probs, axis=-1)
    keluar, sebelumnya = [], None
    for i in ids:
        i = int(i)
        if i != BLANK and i != sebelumnya:
            keluar.append(idx2char[str(i)])
        sebelumnya = i
    return ''.join(keluar)


def jarak_edit(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

**Insight:**

Penyekoran leksikon dilakukan dengan label berbentuk `SparseTensor`, bukan
matriks padat. Pilihan ini bukan detail gaya penulisan: dengan label padat,
`tf.nn.ctc_loss` jatuh ke implementasi berbasis `while_loop` yang pada pengujian
memakan waktu sekitar 2,2 detik per panggilan, dibandingkan 34 milidetik dengan
label jarang — selisih yang menentukan apakah metrik ini layak dihitung setiap
epoch atau tidak.

Kandidat yang secara fisik tidak muat di dalam jumlah frame yang tersedia juga
disaring lebih dahulu, sebab peluangnya nol dan kernel CTC akan menolaknya
dengan galat.

# **EVALUASI PER EPOCH**

Tahap ini mendefinisikan *callback* yang menghitung akurasi produk dan CER pada
subset data validasi setiap beberapa epoch. Subset digunakan agar pelatihan
tidak melambat, sebab penyekoran leksikon jauh lebih mahal daripada perhitungan
loss biasa.

Hasilnya dipecah menurut varian ucapan, karena keberhasilan yang dicari
sifatnya spesifik: model harus membaik pada ucapan sepotong, bukan sekadar
membaik secara rata-rata.

In [ ]:
class EvaluasiProduk(keras.callbacks.Callback):
    """Menghitung akurasi produk dan CER memakai jalur keputusan yang sama dengan produksi."""

    def __init__(self, rows, setiap=2, n_sampel=300, seed=SEED):
        super().__init__()
        self.rows = random.Random(seed).sample(rows, min(n_sampel, len(rows)))
        self.setiap = setiap
        self.riwayat = []

    def _muat(self, r):
        y, _ = librosa.load(os.path.join(WAV_DIR, r['filename']), sr=CFG['sample_rate'])
        return ekstrak_logmel(y)

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.setiap and epoch != 0:
            return
        benar = collections.Counter(); total = collections.Counter()
        cer_atas = cer_bawah = 0
        for r in self.rows:
            probs = self.model({'log_mel': self._muat(r)[None]}, training=False).numpy()[0]
            acuan = r['transcript']
            target = next((p for p in PRODUK if p in acuan), None)
            if target is None:
                continue
            v = r['variant']
            total[v] += 1
            benar[v] += produk_terpilih(probs) == target
            cer_atas += jarak_edit(decode_greedy(probs), acuan)
            cer_bawah += len(acuan)

        n_benar, n_total = sum(benar.values()), sum(total.values())
        akurasi = 100 * n_benar / max(n_total, 1)
        cer = 100 * cer_atas / max(cer_bawah, 1)
        self.riwayat.append({'epoch': epoch + 1, 'akurasi': akurasi, 'cer': cer,
                             **{v: 100 * benar[v] / total[v] for v in total}})
        rincian = '  '.join(f'{v}={100*benar[v]/total[v]:.0f}%' for v in sorted(total))
        print(f'   -> akurasi produk {akurasi:5.1f}%  |  CER {cer:5.1f}%  |  {rincian}')


evaluator = EvaluasiProduk(val_rows, setiap=2, n_sampel=300)

# Ukur kondisi AWAL sebagai garis dasar, sebelum satu langkah pelatihan pun.
print('Garis dasar model v11 pada korpus Tahap 1:')
evaluator.set_model(model)
evaluator.on_epoch_end(-1)
DASAR = evaluator.riwayat[0]
evaluator.riwayat.clear()

**Insight:**

Pengukuran garis dasar dilakukan sebelum pelatihan dimulai. Tanpa angka
pembanding ini, perbaikan setelah fine-tuning tidak dapat diklaim secara jujur —
hasil akhir hanya akan berupa angka tunggal tanpa konteks.

Rincian per varian juga penting karena rata-rata dapat menyesatkan. Model bisa
saja terlihat membaik secara keseluruhan sementara kemampuannya pada ucapan
sepotong justru menurun, dan tanpa pemecahan ini gejala tersebut tidak terlihat.

# **PELATIHAN**

Tahap ini menjalankan fine-tuning. Beberapa pengaman dipasang agar pelatihan
tetap selamat di lingkungan Kaggle yang memiliki batas waktu sesi:

- **Checkpoint** disimpan setiap kali loss validasi membaik, sehingga sesi yang
  terputus tidak menghapus kemajuan.
- **Early stopping** menghentikan pelatihan bila tidak ada perbaikan, sekaligus
  mengembalikan bobot terbaik.
- **Penurunan learning rate** otomatis saat perbaikan melambat.

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(LR_AWAL), loss=ctc_loss)

CKPT = os.path.join(KELUARAN, 'finsense_ctc_v12_terbaik.keras')

callbacks = [
    keras.callbacks.ModelCheckpoint(CKPT, monitor='val_loss', save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE_STOP,
                                  restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                      patience=PATIENCE_LR, min_lr=LR_MINIMUM, verbose=1),
    evaluator,
]

mulai = time.time()
riwayat = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                    callbacks=callbacks, verbose=1)
print(f'\nSelesai dalam {(time.time() - mulai) / 60:.1f} menit')

**Insight:**

Epoch pertama berjalan lebih lambat dibandingkan epoch berikutnya karena
ekstraksi log-mel masih dilakukan dan hasilnya baru ditulis ke cache. Perlambatan
ini normal dan tidak berulang.

Bila loss validasi melonjak pada epoch pertama lalu tidak pernah pulih,
penyebabnya hampir selalu learning rate yang terlalu besar untuk fine-tuning.
Turunkan `LR_AWAL` menjadi `5e-5` dan ulangi.

# **HASIL PELATIHAN**

Tahap ini memvisualisasikan perkembangan loss dan metrik produk selama
pelatihan, serta membandingkannya dengan garis dasar model v11.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

ax[0].plot(riwayat.history['loss'], label='latih')
ax[0].plot(riwayat.history['val_loss'], label='validasi')
ax[0].set_title('Loss CTC'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=.3)

if evaluator.riwayat:
    ep  = [h['epoch'] for h in evaluator.riwayat]
    akr = [h['akurasi'] for h in evaluator.riwayat]
    cer = [h['cer'] for h in evaluator.riwayat]

    ax[1].plot(ep, akr, marker='o', color='#54A24B')
    ax[1].axhline(DASAR['akurasi'], ls='--', color='gray', label=f"v11 ({DASAR['akurasi']:.1f}%)")
    ax[1].set_title('Akurasi produk (validasi)'); ax[1].set_xlabel('epoch')
    ax[1].set_ylabel('%'); ax[1].legend(); ax[1].grid(alpha=.3)

    ax[2].plot(ep, cer, marker='o', color='#E45756')
    ax[2].axhline(DASAR['cer'], ls='--', color='gray', label=f"v11 ({DASAR['cer']:.1f}%)")
    ax[2].set_title('CER transkrip (validasi)'); ax[2].set_xlabel('epoch')
    ax[2].set_ylabel('%'); ax[2].legend(); ax[2].grid(alpha=.3)

plt.tight_layout(); plt.show()

if evaluator.riwayat:
    akhir = max(evaluator.riwayat, key=lambda h: h['akurasi'])
    varian = [v for v in ('produk', 'intent_produk', 'produk_qty', 'full') if v in akhir]
    print(f"{'metrik':22s}{'v11 (sebelum)':>16s}{'v12 (sesudah)':>16s}{'selisih':>12s}")
    print('-' * 66)
    print(f"{'akurasi produk':22s}{DASAR['akurasi']:15.1f}%{akhir['akurasi']:15.1f}%"
          f"{akhir['akurasi'] - DASAR['akurasi']:+11.1f}")
    print(f"{'CER transkrip':22s}{DASAR['cer']:15.1f}%{akhir['cer']:15.1f}%"
          f"{akhir['cer'] - DASAR['cer']:+11.1f}")
    for v in varian:
        if v in DASAR:
            print(f"{'  varian ' + v:22s}{DASAR[v]:15.1f}%{akhir[v]:15.1f}%"
                  f"{akhir[v] - DASAR[v]:+11.1f}")

**Insight:**

Perbandingan disajikan terhadap garis dasar v11 pada korpus yang sama, sehingga
perbaikan dapat dinilai secara langsung. Pemecahan per varian menunjukkan apakah
sasaran utama tercapai: kenaikan terbesar semestinya terjadi pada varian
`produk`, yaitu ucapan yang hanya menyebut nama produk.

Perlu diingat bahwa akurasi pada varian `full` yang sedikit menurun bukan
kegagalan selama penurunannya kecil. Model sedang membagi kapasitasnya untuk
menangani bentuk ucapan yang lebih beragam, dan itu memang pertukaran yang
diinginkan.

# **SIMPAN MODEL**

Tahap ini menyimpan model hasil pelatihan beserta berkas pendamping yang
diperlukan layanan inferensi.

Bentuk masukan dan keluaran model tidak berubah dari v11, sehingga model ini
dapat langsung menggantikannya di `ai-service/serve_model.py` tanpa mengubah
kode layanan.

In [ ]:
JALUR_AKHIR = os.path.join(KELUARAN, 'finsense_ctc_v12.keras')

# Model yang sudah di-compile ikut membawa state optimizer (dua slot momentum
# Adam per parameter), sehingga berkasnya tepat 3x lebih besar tanpa guna apa
# pun saat inferensi: 24.5 MB dibanding 8.2 MB.
#
# Salinan bersih dibangun ulang dari KONFIGURASI arsitekturnya, bukan dengan
# keras.models.clone_model(). clone_model() memanggil compile_from_config()
# sehingga optimizernya ikut terbentuk lagi dan berkasnya tetap 24.5 MB;
# Model.get_config() hanya memuat arsitektur, tidak menyertakan compile.
model_inferensi = keras.Model.from_config(model.get_config())
model_inferensi.set_weights(model.get_weights())
model_inferensi.save(JALUR_AKHIR)

# Kosakata karakter disalin apa adanya: indeksnya harus tetap sama dengan yang
# dipakai lapisan keluaran model.
with open(os.path.join(KELUARAN, 'char_vocab.json'), 'w', encoding='utf-8') as f:
    json.dump(V, f, indent=2)

# Bersihkan cache fitur agar tidak ikut terbawa ke keluaran notebook.
for c in glob.glob(os.path.join(KELUARAN, 'cache_*')):
    try:
        os.remove(c)
    except OSError:
        pass

ukuran = os.path.getsize(JALUR_AKHIR) / 1e6
print(f'model  : {JALUR_AKHIR}  ({ukuran:.1f} MB, tanpa state optimizer)')
print(f'vocab  : {os.path.join(KELUARAN, "char_vocab.json")}')

# Pastikan model yang tersimpan dapat dimuat ulang dan bentuknya tidak berubah.
ulang = keras.models.load_model(JALUR_AKHIR, compile=False)
selisih = np.abs(ulang({'log_mel': contoh_fitur['log_mel'][:1]}, training=False).numpy()
                 - model({'log_mel': contoh_fitur['log_mel'][:1]}, training=False).numpy()).max()
print(f'bobot identik dengan model terlatih — selisih maks {selisih:.2e}')
keluaran = ulang({'log_mel': np.zeros((1, CFG['max_frames'], CFG['n_mels']), np.float32)},
                 training=False)
print(f'muat ulang OK — bentuk keluaran {tuple(keluaran.shape)} '
      f'(harus (1, {CFG["max_frames"] // 2}, {len(char2idx) + 1}))')

**Insight:**

Model tersimpan sekitar 8 MB, bukan 24 MB, karena state optimizer dibuang.
Ukuran ini penting saat model dikirim ke layanan maupun di-commit.

Verifikasi muat ulang dilakukan di dalam notebook, bukan diserahkan ke tahap
deployment. Keluaran model dibandingkan langsung terhadap model yang baru
dilatih, sehingga bila proses penyimpanan merusak sesuatu, hal itu ketahuan di
sini — bukan setelah model terpasang di layanan.

# **LANGKAH SELANJUTNYA**

**1. Kalibrasi ulang ambang keputusan — wajib, jangan dilewati.**

`MODEL_CONFIDENT_SCORE = 0.90` dan `MAX_NLL_PER_CHAR = 6.4` di `serve_model.py`
dikalibrasi terhadap skala keyakinan model v11, bukan besaran universal. Model
baru memiliki sebaran keyakinan yang berbeda, sehingga ambang lama dapat menolak
ucapan yang sah atau menerima ucapan di luar daftar produk.

```bash
cd ai-service
./venv/bin/python eval_tahap0.py    # menghasilkan ambang yang terkalibrasi
```

Gunakan Python milik venv, bukan `python` biasa — `soundfile` dan `tensorflow`
hanya terpasang di sana.

**2. Pasang ke layanan.** Salin `finsense_ctc_v12.keras` ke
`ai-service/model_v11_ctc/`, lalu ubah `MODEL_PATH` di `serve_model.py`.
Berkas `ctc_lexicon.py` tidak perlu disentuh — ia hanya menerima posteriorgram
dan tidak bergantung pada model tertentu.

**3. Bangun ulang korpus dengan model baru.** Forced alignment pada Tahap 1
memakai v11 sebagai penentu batas kata. Setelah v12 tersedia, jalankan ulang
`build_corpus.py` dengan `--model` menunjuk v12, lalu latih sekali lagi. Ini
praktik baku pada ASR: alignment yang lebih baik menghasilkan potongan yang
lebih tepat, yang pada gilirannya menghasilkan model yang lebih baik.

**4. Bila hasilnya belum memuaskan**, urutan yang paling berdampak:

- Tambah data dengan `--no-dedup-augmented` saat membangun korpus. Dataset
  memuat 2600 salinan pitch/noise/speed yang saat ini dilewati; pemisahan
  train/val tetap aman karena dikelompokkan per rekaman dasar.
- Perbesar `max_frames` melampaui 800 agar 410 rekaman yang lebih panjang dari
  8 detik tidak lagi perlu dipangkas.
- Ganti bagian depan CNN menjadi Conv2D di bidang waktu-frekuensi. Conv1D saat
  ini memperlakukan 80 pita mel sebagai 80 kanal terpisah sehingga kehilangan
  informasi kedekatan antar frekuensi.

Perlu dicatat, korpus ini berasal dari penutur yang jumlahnya terbatas. Ragam
bentuk kalimat bertambah, tetapi ragam suara tidak. Batas ini sebaiknya
dinyatakan terbuka dalam laporan, bukan disembunyikan.